In [59]:
import gzip
import os
import re
import shutil
import tempfile
import time
from pathlib import Path, PurePosixPath
from urllib.parse import quote
from zipfile import ZipFile

import pandas as pd
import requests
from remotezip import RemoteZip
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# ============================================================
# USER SETTINGS
# ============================================================

PRIDE_ACCESSIONS = [
    'PXD012460',
'PXD012650',
'PXD014325',
'PXD015297',
'PXD016620',
'PXD016790',
'PXD017601',
'PXD018855',
'PXD018969',
'PXD022985',
'PXD023559',
'PXD025729',
'PXD025792',
'PXD027854',
'PXD033012',
'PXD036036',
'PXD036261',
'PXD036265',
'PXD036770',
'PXD037529',
'PXD039337',
'PXD045017',
'PXD045471',
'PXD045733',
'PXD046940',
'PXD047075',
'PXD052254',
'PXD052837',
'PXD061298',
'PXD064519'
]


OUTPUT_DIR = Path("/Users/mehman/Projects/Estimate_missingness/PRIDE_downloads")

# Also save each calculated DataFrame as PXDxxxxxx_processed.txt
SAVE_PROCESSED_TABLES = True

# Used only if remote ZIP inspection is unavailable.
MAX_FULL_ZIP_DOWNLOAD_GB = 20

REQUEST_TIMEOUT = 300

# UniProt mapping is used only for rows where neither a gene column nor
# a FASTA GN= annotation supplies a gene name.
UNIPROT_MAPPING_BATCH_SIZE = 10000
UNIPROT_MAPPING_TIMEOUT_SECONDS = 900
UNIPROT_POLL_INTERVAL_SECONDS = 3

HYPERGLYCEMIA_FEATURES_FILE = Path(
    "/Users/mehman/Projects/Estimate_missingness/hyperglycemia_features_of_interest.txt"
)

MITOCHONDRIAL_MYOPATHY_FEATURES_FILE = Path(
    "/Users/mehman/Projects/Estimate_missingness/mitochondrial_myopathy_features_of_interest.txt"
)

# Presence is still calculated for every protein row. The feature matching
# below follows the requested manual method and does not filter rows by it.
FEATURE_PRESENCE_THRESHOLD = 0.1
# ============================================================
# CONSTANTS
# ============================================================

PRIDE_API = "https://www.ebi.ac.uk/pride/ws/archive/v3"

PRIDE_DOWNLOADER = (
    "https://www.ebi.ac.uk/pride/"
    "ws/archive-file-downloader/files/s3"
)

METADATA_COLUMNS = [
    "Gene names",
    "Gene matching source",
]

GENE_COLUMN_ALIASES = [
    "gene names",
    "gene name",
    "gene",
    "genes",
    "gene symbol",
    "gene symbols",
    "genesymbol",
    "genesymbols",
]

FASTA_HEADER_COLUMN_ALIASES = {
    "fasta header",
    "fasta headers",
}

MAJORITY_PROTEIN_ID_COLUMN_ALIASES = {
    "majority protein id",
    "majority protein ids",
}

UNIPROT_REST_API = "https://rest.uniprot.org"

UNIPROT_ACCESSION_PATTERN = re.compile(
    r"(?:"
    r"[OPQ][0-9][A-Z0-9]{3}[0-9]"
    r"|"
    r"[A-NR-Z][0-9][A-Z][A-Z0-9]{2}[0-9]"
    r"(?:[A-Z][A-Z0-9]{2}[0-9])?"
    r")"
)

LFQ_PATTERN = re.compile(
    r"^LFQ\s+intensity(?:\s|$)",
    flags=re.IGNORECASE,
)

TARGET_FILES = {
    "proteingroups.txt",
    "proteingroups.txt.gz",
}


# ============================================================
# GENERAL FUNCTIONS
# ============================================================

def create_session():
    """Create a requests session with automatic retries."""

    retry = Retry(
        total=3,
        connect=3,
        read=3,
        status=3,
        backoff_factor=1,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods={"GET", "HEAD"},
    )

    session = requests.Session()

    adapter = HTTPAdapter(
        max_retries=retry,
    )

    session.mount(
        "https://",
        adapter,
    )

    session.mount(
        "http://",
        adapter,
    )

    session.headers.update(
        {
            "User-Agent": (
                "PRIDE-proteinGroups-downloader/1.0"
            )
        }
    )

    return session


def basename(path):
    """Return a filename from a PRIDE or ZIP path."""

    path = str(path).replace(
        "\\",
        "/",
    )

    return PurePosixPath(path).name


def normalize_column(column):
    """Normalize capitalization, whitespace, and punctuation."""

    return re.sub(
        r"[^a-z0-9]+",
        " ",
        str(column).casefold(),
    ).strip()

def find_gene_column(
    columns,
    required=True,
):
    """
    Find the gene-name column using several possible names.

    Examples recognized:

        Gene names
        GENE
        Gene
        Genes
        Gene name
        Gene Symbol
        Gene_symbols
        Gene.names

    The first matching column is returned.
    """

    normalized_columns = {
        normalize_column(column): column
        for column in columns
    }

    # Search the preferred aliases in order.
    for alias in GENE_COLUMN_ALIASES:

        normalized_alias = normalize_column(
            alias
        )

        if normalized_alias in normalized_columns:

            return normalized_columns[
                normalized_alias
            ]

    # Additional fallback for columns such as:
    # "GENE ID", "Gene annotation", etc.
    excluded_gene_columns = {
        "gene ontology",
        "gene ontology id",
        "gene ontology ids",
        "gene description",
    }

    for column in columns:

        normalized = normalize_column(
            column
        )

        if (
            normalized.startswith("gene")
            and normalized
            not in excluded_gene_columns
        ):
            return column

    if required:

        raise ValueError(
            "No gene-name column was found. "
            "Expected a column such as "
            "'Gene names', 'GENE', 'Gene', "
            "'Genes', or 'Gene Symbol'. "
            f"Available columns: {list(columns)}"
        )

    return None


def find_fasta_header_columns(columns):
    """Return all supported FASTA-header columns in their original order."""

    return [
        column
        for column in columns
        if normalize_column(column)
        in FASTA_HEADER_COLUMN_ALIASES
    ]


def find_majority_protein_id_column(columns):
    """Find Majority protein ID/IDs despite spacing or punctuation."""

    for column in columns:
        if (
            normalize_column(column)
            in MAJORITY_PROTEIN_ID_COLUMN_ALIASES
        ):
            return column

    return None


def extract_api_files(payload):
    """Extract file records from different PRIDE response structures."""

    if isinstance(payload, list):
        return payload

    if not isinstance(payload, dict):
        return []

    for key in [
        "files",
        "content",
        "results",
    ]:
        value = payload.get(key)

        if isinstance(value, list):
            return value

    embedded = payload.get("_embedded")

    if isinstance(embedded, dict):
        for value in embedded.values():
            if isinstance(value, list):
                return value

    return []


# ============================================================
# PRIDE FILE DISCOVERY
# ============================================================

def get_pride_files(
    accession,
    session,
):
    """
    Get all public files belonging to one PRIDE accession.

    PRIDE may ignore the requested page size or impose its own
    maximum page size. Therefore, continue requesting pages until
    PRIDE returns an empty page.
    """

    all_files = []
    seen = set()

    for page in range(1000):

        response = session.get(
            (
                f"{PRIDE_API}/projects/"
                f"{accession}/files"
            ),
            params={
                "page": page,
            },
            timeout=REQUEST_TIMEOUT,
        )

        response.raise_for_status()

        batch = extract_api_files(
            response.json()
        )

        print(
            f"  PRIDE page {page}: "
            f"{len(batch)} files"
        )

        # An empty response means that all pages were read.
        if not batch:
            break

        new_files = 0

        for record in batch:

            file_name = str(
                record.get("fileName") or ""
            ).strip()

            file_id = str(
                record.get("accession")
                or file_name
            )

            if not file_name:
                continue

            if file_id in seen:
                continue

            seen.add(file_id)
            all_files.append(record)
            new_files += 1

        # Safety check in case PRIDE unexpectedly returns
        # the same page repeatedly.
        if new_files == 0:
            break

    else:
        raise RuntimeError(
            f"PRIDE pagination exceeded 1000 pages "
            f"for {accession}."
        )

    return all_files


def get_download_url(
    accession,
    record,
):
    """Extract a usable HTTPS download URL from a PRIDE file record."""

    locations = (
        record.get("publicFileLocations")
        or []
    )

    # Prefer an existing HTTP/HTTPS location.
    for location in locations:

        value = str(
            location.get("value") or ""
        ).strip()

        if value.startswith(
            ("https://", "http://")
        ):
            return value

    # Convert the public PRIDE FTP link to HTTPS.
    for location in locations:

        value = str(
            location.get("value") or ""
        ).strip()

        if value.startswith(
            "ftp://ftp.pride.ebi.ac.uk/"
        ):
            return value.replace(
                "ftp://ftp.pride.ebi.ac.uk/",
                "https://ftp.pride.ebi.ac.uk/",
                1,
            )

    # Final fallback: PRIDE streaming downloader.
    file_name = record["fileName"]

    return (
        f"{PRIDE_DOWNLOADER}/"
        f"{quote(accession, safe='')}/"
        f"{quote(file_name, safe='/')}"
    )


def get_file_category(record):
    """Return the PRIDE file category."""

    category = (
        record.get("fileCategory")
        or {}
    )

    if isinstance(category, dict):
        return str(
            category.get("value")
            or category.get("name")
            or ""
        ).upper()

    return str(category).upper()


def get_file_size(record):
    """Return file size in bytes."""

    try:
        return int(
            record.get("fileSizeBytes")
            or 0
        )
    except (TypeError, ValueError):
        return 0


# ============================================================
# HEADER INSPECTION
# ============================================================

def parse_header(line):
    """Decode a proteinGroups.txt header."""

    text = line.decode(
        "utf-8-sig",
        errors="replace",
    )

    return [
        column.strip()
        for column in text.rstrip(
            "\r\n"
        ).split("\t")
    ]


def inspect_header(header):
    """
    Check for a gene source and identify LFQ columns.

    A gene column, FASTA header, or Majority protein IDs column is
    sufficient. Protein IDs are not required.
    """

    gene_column = find_gene_column(
        header,
        required=False,
    )

    fasta_header_columns = (
        find_fasta_header_columns(
            header
        )
    )

    majority_protein_id_column = (
        find_majority_protein_id_column(
            header
        )
    )

    lfq_columns = [
        column
        for column in header
        if LFQ_PATTERN.match(
            str(column).strip()
        )
    ]

    valid = (
        (
            gene_column is not None
            or len(fasta_header_columns) > 0
            or majority_protein_id_column is not None
        )
        and len(lfq_columns) > 0
    )

    return valid, lfq_columns


def read_direct_header(
    url,
    session,
    gzipped=False,
):
    """Read only the header of a direct PRIDE file."""

    headers = {}

    if not gzipped:
        headers["Range"] = (
            "bytes=0-4194303"
        )

    with session.get(
        url,
        headers=headers,
        stream=True,
        timeout=REQUEST_TIMEOUT,
    ) as response:

        response.raise_for_status()
        response.raw.decode_content = True

        if gzipped:

            with gzip.GzipFile(
                fileobj=response.raw
            ) as stream:

                line = stream.readline(
                    4 * 1024 * 1024
                )

        else:
            line = response.raw.readline(
                4 * 1024 * 1024
            )

    return parse_header(line)


def read_zip_member_header(
    archive,
    member,
):
    """Read a proteinGroups header from a ZIP member."""

    with archive.open(member) as stream:

        if basename(member).casefold().endswith(
            ".gz"
        ):

            with gzip.GzipFile(
                fileobj=stream
            ) as decompressed:

                line = decompressed.readline(
                    4 * 1024 * 1024
                )

        else:
            line = stream.readline(
                4 * 1024 * 1024
            )

    return parse_header(line)


# ============================================================
# DOWNLOAD AND EXTRACTION
# ============================================================

def copy_stream(
    source,
    destination,
):
    """Safely copy a binary stream to a file."""

    temporary = destination.with_name(
        destination.name + ".part"
    )

    temporary.unlink(
        missing_ok=True
    )

    try:

        with temporary.open("wb") as output:

            shutil.copyfileobj(
                source,
                output,
                length=4 * 1024 * 1024,
            )

        if temporary.stat().st_size == 0:
            raise IOError(
                "The extracted file is empty."
            )

        os.replace(
            temporary,
            destination,
        )

    finally:
        temporary.unlink(
            missing_ok=True
        )


def download_file(
    url,
    destination,
    session,
):
    """Download a complete file."""

    temporary = destination.with_name(
        destination.name + ".part"
    )

    temporary.unlink(
        missing_ok=True
    )

    try:

        with session.get(
            url,
            stream=True,
            timeout=REQUEST_TIMEOUT,
        ) as response:

            response.raise_for_status()

            with temporary.open(
                "wb"
            ) as output:

                for chunk in response.iter_content(
                    chunk_size=4 * 1024 * 1024
                ):
                    if chunk:
                        output.write(chunk)

        if temporary.stat().st_size == 0:
            raise IOError(
                "The downloaded file is empty."
            )

        os.replace(
            temporary,
            destination,
        )

    finally:
        temporary.unlink(
            missing_ok=True
        )


def extract_direct_file(
    url,
    output_file,
    session,
    gzipped=False,
):
    """Download a direct proteinGroups file."""

    with session.get(
        url,
        stream=True,
        timeout=REQUEST_TIMEOUT,
    ) as response:

        response.raise_for_status()
        response.raw.decode_content = True

        if gzipped:

            with gzip.GzipFile(
                fileobj=response.raw
            ) as stream:

                copy_stream(
                    stream,
                    output_file,
                )

        else:
            copy_stream(
                response.raw,
                output_file,
            )


def extract_zip_member(
    archive,
    member,
    output_file,
):
    """Extract proteinGroups.txt from an opened ZIP archive."""

    with archive.open(member) as stream:

        if basename(member).casefold().endswith(
            ".gz"
        ):

            with gzip.GzipFile(
                fileobj=stream
            ) as decompressed:

                copy_stream(
                    decompressed,
                    output_file,
                )

        else:
            copy_stream(
                stream,
                output_file,
            )


# ============================================================
# FIND proteinGroups.txt
# ============================================================

def find_and_download_proteingroups(
    accession,
    output_file,
    session,
):
    """Find the best LFQ-containing proteinGroups.txt for one accession."""

    records = get_pride_files(
        accession,
        session,
    )

    if not records:
        raise FileNotFoundError(
            "PRIDE returned no public files."
        )

    print(
        f"  PRIDE files: {len(records)}"
    )

    candidates = []

    with tempfile.TemporaryDirectory(
        prefix=f"{accession}_"
    ) as temporary_directory:

        temporary_directory = Path(
            temporary_directory
        )

        # ----------------------------------------------------
        # Direct proteinGroups.txt files
        # ----------------------------------------------------

        for record in records:

            file_name = record["fileName"]

            if (
                basename(file_name).casefold()
                not in TARGET_FILES
            ):
                continue

            url = get_download_url(
                accession,
                record,
            )

            gzipped = (
                basename(file_name)
                .casefold()
                .endswith(".gz")
            )

            try:

                header = read_direct_header(
                    url,
                    session,
                    gzipped=gzipped,
                )

                valid, lfq_columns = (
                    inspect_header(header)
                )

                if valid:
                    candidates.append(
                        {
                            "type": "direct",
                            "record": record,
                            "url": url,
                            "member": None,
                            "local_zip": None,
                            "lfq_count": len(
                                lfq_columns
                            ),
                        }
                    )

            except Exception as error:

                print(
                    f"  Could not inspect "
                    f"{file_name}: {error}"
                )

        # ----------------------------------------------------
        # ZIP archives
        # ----------------------------------------------------

        zip_records = [
            record
            for record in records
            if basename(
                record["fileName"]
            ).casefold().endswith(".zip")
        ]

        # Search likely MaxQuant/result ZIP files first.
        zip_records.sort(
            key=lambda record: (
                0
                if "maxquant"
                in record["fileName"].casefold()
                else 1,
                0
                if get_file_category(record)
                in {"SEARCH", "RESULT"}
                else 1,
                get_file_size(record),
            )
        )

        for zip_number, record in enumerate(
            zip_records,
            start=1,
        ):

            zip_name = record["fileName"]

            url = get_download_url(
                accession,
                record,
            )

            print(
                f"  Inspecting ZIP "
                f"{zip_number}/{len(zip_records)}: "
                f"{zip_name}"
            )

            try:

                # Inspect and read members without downloading
                # the entire ZIP archive.
                with RemoteZip(
                    url,
                    session=session,
                    timeout=REQUEST_TIMEOUT,
                    initial_buffer_size=1024 * 1024,
                    support_suffix_range=False,
                ) as archive:

                    members = [
                        info.filename
                        for info
                        in archive.infolist()
                        if not info.is_dir()
                        and basename(
                            info.filename
                        ).casefold()
                        in TARGET_FILES
                    ]

                    for member in members:

                        header = (
                            read_zip_member_header(
                                archive,
                                member,
                            )
                        )

                        valid, lfq_columns = (
                            inspect_header(header)
                        )

                        if valid:
                            candidates.append(
                                {
                                    "type": "remote_zip",
                                    "record": record,
                                    "url": url,
                                    "member": member,
                                    "local_zip": None,
                                    "lfq_count": len(
                                        lfq_columns
                                    ),
                                }
                            )

            except Exception as remote_error:

                # If byte-range access does not work,
                # download the complete ZIP as a fallback.
                file_size = get_file_size(
                    record
                )

                maximum_size = (
                    MAX_FULL_ZIP_DOWNLOAD_GB
                    * 1024**3
                )

                if (
                    file_size
                    and file_size > maximum_size
                ):
                    print(
                        f"  Skipping full ZIP fallback: "
                        f"{file_size / 1024**3:.2f} GB"
                    )
                    continue

                print(
                    "  Remote ZIP inspection failed; "
                    "trying full ZIP download."
                )

                local_zip = (
                    temporary_directory
                    / f"archive_{zip_number}.zip"
                )

                try:

                    download_file(
                        url,
                        local_zip,
                        session,
                    )

                    with ZipFile(
                        local_zip
                    ) as archive:

                        members = [
                            info.filename
                            for info
                            in archive.infolist()
                            if not info.is_dir()
                            and basename(
                                info.filename
                            ).casefold()
                            in TARGET_FILES
                        ]

                        for member in members:

                            header = (
                                read_zip_member_header(
                                    archive,
                                    member,
                                )
                            )

                            valid, lfq_columns = (
                                inspect_header(header)
                            )

                            if valid:
                                candidates.append(
                                    {
                                        "type": "local_zip",
                                        "record": record,
                                        "url": url,
                                        "member": member,
                                        "local_zip": local_zip,
                                        "lfq_count": len(
                                            lfq_columns
                                        ),
                                    }
                                )

                except Exception as full_error:

                    print(
                        f"  ZIP failed: "
                        f"{full_error}"
                    )

        if not candidates:
            raise FileNotFoundError(
                "No valid proteinGroups.txt was found."
            )

        # Require an LFQ-containing proteinGroups.txt.
        lfq_candidates = [
            candidate
            for candidate in candidates
            if candidate["lfq_count"] > 0
        ]

        if not lfq_candidates:
            raise ValueError(
                "proteinGroups.txt was found, but it "
                "does not contain LFQ intensity columns."
            )

        # Select the file containing the largest number of LFQ columns.
        selected = max(
            lfq_candidates,
            key=lambda candidate: (
                candidate["lfq_count"],
                candidate["type"] == "direct",
            ),
        )

        selected_name = (
            selected["member"]
            or selected["record"]["fileName"]
        )

        print(
            f"  Selected: {selected_name}"
        )

        print(
            f"  LFQ columns: "
            f"{selected['lfq_count']}"
        )

        # ----------------------------------------------------
        # Extract selected file
        # ----------------------------------------------------

        if selected["type"] == "direct":

            gzipped = (
                basename(
                    selected["record"]["fileName"]
                )
                .casefold()
                .endswith(".gz")
            )

            extract_direct_file(
                selected["url"],
                output_file,
                session,
                gzipped=gzipped,
            )

        elif selected["type"] == "local_zip":

            with ZipFile(
                selected["local_zip"]
            ) as archive:

                extract_zip_member(
                    archive,
                    selected["member"],
                    output_file,
                )

        else:

            try:

                with RemoteZip(
                    selected["url"],
                    session=session,
                    timeout=REQUEST_TIMEOUT,
                    initial_buffer_size=1024 * 1024,
                    support_suffix_range=False,
                ) as archive:

                    extract_zip_member(
                        archive,
                        selected["member"],
                        output_file,
                    )

            except Exception:

                # Extraction fallback: download the full selected ZIP.
                local_zip = (
                    temporary_directory
                    / "selected_archive.zip"
                )

                download_file(
                    selected["url"],
                    local_zip,
                    session,
                )

                with ZipFile(
                    local_zip
                ) as archive:

                    extract_zip_member(
                        archive,
                        selected["member"],
                        output_file,
                    )


# ============================================================
# DATAFRAME AND PRESENCE CALCULATION
# ============================================================

def create_presence_dataframe(
    protein_groups_file,
    session=None,
    uniprot_gene_cache=None,
):
    """
    Read the gene and LFQ columns and calculate protein presence.

    Gene names are collected from a supported gene column and from
    every GN= entry in Fasta header/Fasta headers columns. For a row
    where neither source provides a gene, Majority protein IDs are
    converted to gene names through the UniProt ID Mapping service.

    Presence =
        number of non-missing, non-zero LFQ values
        divided by
        total number of LFQ columns
    """

    # Read only the header first.
    header = list(
        pd.read_csv(
            protein_groups_file,
            sep="\t",
            nrows=0,
        ).columns
    )

    # Automatically detect all available gene sources.
    gene_column = find_gene_column(
        header,
        required=False,
    )

    fasta_header_columns = (
        find_fasta_header_columns(
            header
        )
    )

    majority_protein_id_column = (
        find_majority_protein_id_column(
            header
        )
    )

    if (
        gene_column is None
        and not fasta_header_columns
        and majority_protein_id_column is None
    ):
        raise ValueError(
            "No usable gene source was found. "
            "Expected a gene column such as 'Gene names' or "
            "a column named 'Fasta header'/'Fasta headers' or "
            "'Majority protein IDs'. "
            f"Available columns: {header}"
        )

    if gene_column is not None:
        print(
            f"  Gene column detected: "
            f"{gene_column}"
        )

    if fasta_header_columns:
        print(
            "  FASTA header column(s) detected: "
            + ", ".join(
                str(column)
                for column in fasta_header_columns
            )
        )

    if majority_protein_id_column is not None:
        print(
            "  Majority protein IDs column detected: "
            f"{majority_protein_id_column}"
        )

    lfq_columns = [
        column
        for column in header
        if LFQ_PATTERN.match(
            str(column).strip()
        )
    ]

    if not lfq_columns:

        raise ValueError(
            "No LFQ intensity columns were found."
        )

    # Protein IDs are not loaded. Majority protein IDs are loaded only
    # because they provide the final gene-name fallback.
    gene_source_columns = (
        ([gene_column] if gene_column is not None else [])
        + fasta_header_columns
        + (
            [majority_protein_id_column]
            if majority_protein_id_column is not None
            else []
        )
    )

    columns_to_read = list(
        dict.fromkeys(
            gene_source_columns
            + lfq_columns
        )
    )

    dataframe = pd.read_csv(
        protein_groups_file,
        sep="\t",
        usecols=columns_to_read,
        low_memory=False,
    )

    # Build one consistent gene field and record how each row obtained
    # its gene names.
    gene_information = dataframe.apply(
        lambda row: collect_row_gene_information(
            row=row,
            gene_column=gene_column,
            fasta_header_columns=(
                fasta_header_columns
            ),
        ),
        axis=1,
    )

    dataframe["Gene names"] = gene_information.map(
        lambda value: value[0]
    )

    dataframe["Gene matching source"] = (
        gene_information.map(
            lambda value: value[1]
        )
    )

    # Use Majority protein IDs only for rows that still have no gene.
    apply_majority_protein_id_fallback(
        dataframe=dataframe,
        majority_protein_id_column=(
            majority_protein_id_column
        ),
        session=session,
        uniprot_gene_cache=uniprot_gene_cache,
    )

    # Ensure LFQ values are numeric.
    lfq_numeric = dataframe[
        lfq_columns
    ].apply(
        pd.to_numeric,
        errors="coerce",
    )

    # Missing values and zeros are both considered absent.
    present = (
        lfq_numeric.notna()
        & lfq_numeric.ne(0)
    )

    presence_count = present.sum(
        axis=1
    )

    total_lfq_columns = len(
        lfq_columns
    )

    dataframe["Presence_count"] = (
        presence_count
    )

    dataframe["Total_LFQ_columns"] = (
        total_lfq_columns
    )

    dataframe["Presence"] = (
        presence_count
        / total_lfq_columns
    )

    dataframe["Presence_percent"] = (
        dataframe["Presence"]
        * 100
    )

    final_columns = (
        [
            "Gene names",
            "Gene matching source",
        ]
        + lfq_columns
        + [
            "Presence_count",
            "Total_LFQ_columns",
            "Presence",
            "Presence_percent",
        ]
    )

    return dataframe[
        final_columns
    ]

# ============================================================
# FEATURE-SET MISSINGNESS
# ============================================================

def read_feature_genes(file_path):
    """
    Read unique gene names from a text file.

    Supported examples:

        TP53
        BRCA1
        BRCA2

    or:

        TP53;BRCA1;BRCA2

    or comma-/tab-separated gene names.
    """

    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(
            f"Feature file was not found: {file_path}"
        )

    header_names = {
        "gene",
        "genes",
        "gene name",
        "gene names",
        "feature",
        "features",
        "gene symbol",
        "gene symbols",
    }

    genes = []
    seen = set()

    text = file_path.read_text(
        encoding="utf-8-sig",
    )

    for line in text.splitlines():

        # Remove comments starting with #
        line = line.split(
            "#",
            maxsplit=1,
        )[0].strip()

        if not line:
            continue

        # Accept one gene per line as well as comma-, semicolon-,
        # or tab-separated genes.
        entries = re.split(
            r"[;,\t]+",
            line,
        )

        for entry in entries:

            gene = entry.strip()

            if not gene:
                continue

            if gene.casefold() in header_names:
                continue

            normalized_gene = gene.upper()

            if normalized_gene not in seen:

                seen.add(
                    normalized_gene
                )

                genes.append(
                    gene
                )

    if not genes:
        raise ValueError(
            f"No gene names were found in {file_path}"
        )

    return genes


def split_gene_names(gene_names_value):
    """
    Split a MaxQuant Gene names cell.

    A cell such as:

        GENE1;GENE2;GENE3

    becomes:

        {"GENE1", "GENE2", "GENE3"}
    """

    if pd.isna(gene_names_value):
        return set()

    genes = set()

    for gene in str(
        gene_names_value
    ).split(";"):

        gene = gene.strip()

        if gene:
            genes.add(
                gene.upper()
            )

    return genes


def extract_fasta_gene_names(fasta_header_value):
    """Extract every gene symbol following GN= in a FASTA header cell."""

    if pd.isna(fasta_header_value):
        return []

    return re.findall(
        r"\bGN=([^\s;]+)",
        str(fasta_header_value),
        flags=re.IGNORECASE,
    )


def collect_row_gene_information(
    row,
    gene_column,
    fasta_header_columns,
):
    """Return combined gene names and their source for one row."""

    genes = []
    seen = set()
    sources = []

    if (
        gene_column is not None
        and not pd.isna(row[gene_column])
    ):
        gene_column_supplied_a_gene = False

        for gene in str(
            row[gene_column]
        ).split(";"):
            gene = gene.strip()
            normalized_gene = gene.upper()

            if (
                normalized_gene
                and normalized_gene not in seen
            ):
                seen.add(normalized_gene)
                genes.append(gene)
                gene_column_supplied_a_gene = True

        if gene_column_supplied_a_gene:
            sources.append(
                f"Gene column ({gene_column})"
            )

    fasta_columns_with_genes = []

    for fasta_column in fasta_header_columns:
        fasta_genes = extract_fasta_gene_names(
            row[fasta_column]
        )

        if fasta_genes:
            fasta_columns_with_genes.append(
                str(fasta_column)
            )

        for gene in fasta_genes:
            gene = gene.strip()
            normalized_gene = gene.upper()

            if (
                normalized_gene
                and normalized_gene not in seen
            ):
                seen.add(normalized_gene)
                genes.append(gene)

    if not genes:
        return pd.NA, "Unresolved"

    if fasta_columns_with_genes:
        sources.append(
            "FASTA GN= ("
            + ", ".join(
                fasta_columns_with_genes
            )
            + ")"
        )

    return ";".join(genes), " + ".join(sources)


def combine_row_gene_names(
    row,
    gene_column,
    fasta_header_columns,
):
    """Backward-compatible wrapper returning only combined gene names."""

    return collect_row_gene_information(
        row=row,
        gene_column=gene_column,
        fasta_header_columns=fasta_header_columns,
    )[0]


def extract_uniprot_accessions(protein_id_value):
    """
    Extract canonical UniProt accessions from a Majority protein IDs cell.

    Isoform suffixes are removed, for example P06310-1 becomes P06310.
    Both plain accessions and sp|P06310|ENTRY-style identifiers work.
    """

    if pd.isna(protein_id_value):
        return []

    accessions = []
    seen = set()

    for value in str(protein_id_value).split(";"):
        value = value.strip()

        if not value:
            continue

        if "|" in value:
            parts = value.split("|")

            if len(parts) >= 2:
                value = parts[1]

        value = value.split()[0]
        value = re.sub(
            r"-\d+$",
            "",
            value,
        ).upper()

        if not UNIPROT_ACCESSION_PATTERN.fullmatch(
            value
        ):
            continue

        if value not in seen:
            seen.add(value)
            accessions.append(value)

    return accessions


def mapped_gene_values(mapped_value):
    """Normalize a UniProt mapping result into a list of gene names."""

    if isinstance(mapped_value, str):
        return [mapped_value]

    if isinstance(mapped_value, list):
        genes = []

        for value in mapped_value:
            genes.extend(
                mapped_gene_values(value)
            )

        return genes

    if isinstance(mapped_value, dict):
        for key in [
            "value",
            "geneName",
            "id",
        ]:
            if key in mapped_value:
                return mapped_gene_values(
                    mapped_value[key]
                )

    return []


def map_uniprot_accessions_to_gene_names(
    accessions,
    session,
    cache=None,
):
    """Batch-map UniProtKB accessions to UniProt Gene_Name values."""

    if cache is None:
        cache = {}

    accessions = list(
        dict.fromkeys(accessions)
    )

    accessions_to_query = [
        accession
        for accession in accessions
        if accession not in cache
    ]

    for start in range(
        0,
        len(accessions_to_query),
        UNIPROT_MAPPING_BATCH_SIZE,
    ):
        batch = accessions_to_query[
            start:
            start + UNIPROT_MAPPING_BATCH_SIZE
        ]

        response = session.post(
            f"{UNIPROT_REST_API}/idmapping/run",
            data={
                "from": "UniProtKB_AC-ID",
                "to": "Gene_Name",
                "ids": ",".join(batch),
            },
            timeout=REQUEST_TIMEOUT,
        )

        response.raise_for_status()
        job_id = response.json().get(
            "jobId"
        )

        if not job_id:
            raise RuntimeError(
                "UniProt did not return an ID-mapping job ID."
            )

        deadline = (
            time.monotonic()
            + UNIPROT_MAPPING_TIMEOUT_SECONDS
        )

        while True:
            status_response = session.get(
                (
                    f"{UNIPROT_REST_API}"
                    f"/idmapping/status/{job_id}"
                ),
                timeout=REQUEST_TIMEOUT,
            )

            status_response.raise_for_status()
            status_payload = (
                status_response.json()
            )
            job_status = status_payload.get(
                "jobStatus"
            )

            if job_status in {
                "NEW",
                "RUNNING",
            }:
                if time.monotonic() >= deadline:
                    raise TimeoutError(
                        "UniProt ID mapping exceeded "
                        f"{UNIPROT_MAPPING_TIMEOUT_SECONDS} seconds."
                    )

                time.sleep(
                    UNIPROT_POLL_INTERVAL_SECONDS
                )
                continue

            if job_status == "FAILED":
                raise RuntimeError(
                    "UniProt ID mapping job failed."
                )

            break

        results_response = session.get(
            (
                f"{UNIPROT_REST_API}"
                f"/idmapping/stream/{job_id}"
            ),
            params={
                "format": "json",
            },
            timeout=REQUEST_TIMEOUT,
        )

        results_response.raise_for_status()
        results_payload = (
            results_response.json()
        )

        for accession in batch:
            cache.setdefault(
                accession,
                [],
            )

        for result in results_payload.get(
            "results",
            [],
        ):
            accession = str(
                result.get("from", "")
            ).upper()

            if accession not in cache:
                continue

            for gene in mapped_gene_values(
                result.get("to")
            ):
                gene = str(gene).strip()

                if (
                    gene
                    and gene not in cache[accession]
                ):
                    cache[accession].append(gene)

    return {
        accession: list(
            cache.get(accession, [])
        )
        for accession in accessions
    }


def apply_majority_protein_id_fallback(
    dataframe,
    majority_protein_id_column,
    session=None,
    uniprot_gene_cache=None,
):
    """Fill unresolved gene rows using Majority protein IDs and UniProt."""

    if majority_protein_id_column is None:
        return

    unresolved_indexes = dataframe.index[
        dataframe["Gene names"].isna()
    ]

    row_accessions = {}

    for index in unresolved_indexes:
        accessions = extract_uniprot_accessions(
            dataframe.at[
                index,
                majority_protein_id_column,
            ]
        )

        if accessions:
            row_accessions[index] = accessions
        elif not pd.isna(
            dataframe.at[
                index,
                majority_protein_id_column,
            ]
        ):
            dataframe.at[
                index,
                "Gene matching source",
            ] = (
                "Majority protein IDs -> "
                "no valid UniProt accession"
            )

    unique_accessions = list(
        dict.fromkeys(
            accession
            for accessions in row_accessions.values()
            for accession in accessions
        )
    )

    if not unique_accessions:
        return

    print(
        "  Mapping Majority protein IDs through UniProt: "
        f"{len(unique_accessions):,} unique accessions"
    )

    owns_session = session is None

    if owns_session:
        session = create_session()

    try:
        accession_to_genes = (
            map_uniprot_accessions_to_gene_names(
                accessions=unique_accessions,
                session=session,
                cache=uniprot_gene_cache,
            )
        )

    except Exception as error:
        print(
            "  WARNING: UniProt gene mapping failed: "
            f"{error}"
        )

        for index in row_accessions:
            dataframe.at[
                index,
                "Gene matching source",
            ] = (
                "Majority protein IDs -> "
                "UniProt mapping failed"
            )

        return

    finally:
        if owns_session:
            session.close()

    resolved_row_count = 0

    for index, accessions in row_accessions.items():
        genes = []
        seen = set()

        for accession in accessions:
            for gene in accession_to_genes.get(
                accession,
                [],
            ):
                normalized_gene = gene.upper()

                if normalized_gene not in seen:
                    seen.add(normalized_gene)
                    genes.append(gene)

        if genes:
            dataframe.at[
                index,
                "Gene names",
            ] = ";".join(genes)

            dataframe.at[
                index,
                "Gene matching source",
            ] = (
                "Majority protein IDs -> "
                "UniProt Gene_Name"
            )

            resolved_row_count += 1

        else:
            dataframe.at[
                index,
                "Gene matching source",
            ] = (
                "Majority protein IDs -> "
                "no UniProt Gene_Name"
            )

    print(
        "  Rows resolved from Majority protein IDs: "
        f"{resolved_row_count:,}/{len(row_accessions):,}"
    )


def calculate_feature_missingness(
    dataframe,
    feature_genes,
    presence_threshold=0.1,
):
    """
    Calculate feature missingness for one PRIDE dataset.

    A feature gene is considered found when:

      1. It occurs in the Gene names field.
      2. Multiple names separated by ';' are checked independently.

    Matching intentionally follows the manual calculation exactly:

        Genes_present = (
            dataframe["Gene names"]
            .dropna()
            .str.split(";")
            .explode()
            .str.strip()
            .dropna()
            .unique()
        )

        intersection = set(Genes_present) & set(feature_genes)

    The presence_threshold argument is retained so existing calls do not
    break, but it is not used to filter genes during this exact matching.

    Missingness percentage:

        missing target genes
        -------------------- × 100
        total target genes
    """

    target_genes = set(
        feature_genes
    )

    if not target_genes:
        raise ValueError(
            "The feature-gene list is empty."
        )

    Genes_present = (
        dataframe["Gene names"]
        .dropna()
        .str.split(";")
        .explode()
        .str.strip()
        .dropna()
        .unique()
    )

    intersection = (
        set(Genes_present)
        & set(feature_genes)
    )

    found_genes = intersection

    missing_genes = (
        target_genes
        - found_genes
    )

    total_gene_count = len(
        target_genes
    )

    found_gene_count = len(
        found_genes
    )

    missing_gene_count = len(
        missing_genes
    )

    missingness_percentage = (
        missing_gene_count
        / total_gene_count
        * 100
    )

    found_gene_names = sorted(
        found_genes
    )

    missing_gene_names = sorted(
        missing_genes
    )

    return {
        "total_feature_genes": total_gene_count,
        "found_feature_genes": found_gene_count,
        "missing_feature_genes": missing_gene_count,
        "missingness_percent": missingness_percentage,
        "found_gene_names": ";".join(
            found_gene_names
        ),
        "missing_gene_names": ";".join(
            missing_gene_names
        ),
    }

# ============================================================
# RUN PIPELINE
# ============================================================

# ============================================================
# RUN PIPELINE
# ============================================================

def main():

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    accessions = list(
        dict.fromkeys(
            accession.strip().upper()
            for accession in PRIDE_ACCESSIONS
            if accession.strip()
        )
    )

    # Read the two feature lists once.
    hyperglycemia_features = (
        read_feature_genes(
            HYPERGLYCEMIA_FEATURES_FILE
        )
    )

    mitochondrial_myopathy_features = (
        read_feature_genes(
            MITOCHONDRIAL_MYOPATHY_FEATURES_FILE
        )
    )

    print(
        "Hyperglycemia feature genes: "
        f"{len(hyperglycemia_features)}"
    )

    print(
        "Mitochondrial myopathy feature genes: "
        f"{len(mitochondrial_myopathy_features)}"
    )

    session = create_session()

    # Each accession has its own protein-level DataFrame.
    dataframes = {}

    # Reuse successful UniProt mappings across all PRIDE datasets.
    uniprot_gene_cache = {}

    # One row per accession containing the two missingness scores.
    missingness_results = []

    try:

        for number, accession in enumerate(
            accessions,
            start=1,
        ):

            print(
                f"\n[{number}/{len(accessions)}] "
                f"{accession}"
            )

            original_file = (
                OUTPUT_DIR
                / f"{accession}.txt"
            )

            try:

                # Existing download logic remains unchanged.
                find_and_download_proteingroups(
                    accession,
                    original_file,
                    session,
                )

                # Existing presence calculation remains unchanged.
                dataframe = (
                    create_presence_dataframe(
                        protein_groups_file=(
                            original_file
                        ),
                        session=session,
                        uniprot_gene_cache=(
                            uniprot_gene_cache
                        ),
                    )
                )

                dataframes[accession] = (
                    dataframe
                )

                if SAVE_PROCESSED_TABLES:

                    processed_file = (
                        OUTPUT_DIR
                        / (
                            f"{accession}"
                            "_processed.txt"
                        )
                    )

                    dataframe.to_csv(
                        processed_file,
                        sep="\t",
                        index=False,
                    )

                # --------------------------------------------
                # Hyperglycemia feature missingness
                # --------------------------------------------

                hyperglycemia_score = (
                    calculate_feature_missingness(
                        dataframe=dataframe,
                        feature_genes=(
                            hyperglycemia_features
                        ),
                        presence_threshold=(
                            FEATURE_PRESENCE_THRESHOLD
                        ),
                    )
                )

                # --------------------------------------------
                # Mitochondrial-myopathy feature missingness
                # --------------------------------------------

                mitochondrial_score = (
                    calculate_feature_missingness(
                        dataframe=dataframe,
                        feature_genes=(
                            mitochondrial_myopathy_features
                        ),
                        presence_threshold=(
                            FEATURE_PRESENCE_THRESHOLD
                        ),
                    )
                )

                missingness_results.append(
                    {
                        "PRIDE_accession": accession,
                        "presence_threshold": (
                            FEATURE_PRESENCE_THRESHOLD
                        ),

                        "hyperglycemia_total_genes": (
                            hyperglycemia_score[
                                "total_feature_genes"
                            ]
                        ),
                        "hyperglycemia_found_genes": (
                            hyperglycemia_score[
                                "found_feature_genes"
                            ]
                        ),
                        "hyperglycemia_missing_genes": (
                            hyperglycemia_score[
                                "missing_feature_genes"
                            ]
                        ),
                        "hyperglycemia_missingness_percent": (
                            hyperglycemia_score[
                                "missingness_percent"
                            ]
                        ),
                        "hyperglycemia_found_gene_names": (
                            hyperglycemia_score[
                                "found_gene_names"
                            ]
                        ),
                        "hyperglycemia_missing_gene_names": (
                            hyperglycemia_score[
                                "missing_gene_names"
                            ]
                        ),

                        "mitochondrial_myopathy_total_genes": (
                            mitochondrial_score[
                                "total_feature_genes"
                            ]
                        ),
                        "mitochondrial_myopathy_found_genes": (
                            mitochondrial_score[
                                "found_feature_genes"
                            ]
                        ),
                        "mitochondrial_myopathy_missing_genes": (
                            mitochondrial_score[
                                "missing_feature_genes"
                            ]
                        ),
                        "mitochondrial_myopathy_missingness_percent": (
                            mitochondrial_score[
                                "missingness_percent"
                            ]
                        ),
                        "mitochondrial_myopathy_found_gene_names": (
                            mitochondrial_score[
                                "found_gene_names"
                            ]
                        ),
                        "mitochondrial_myopathy_missing_gene_names": (
                            mitochondrial_score[
                                "missing_gene_names"
                            ]
                        ),
                    }
                )

                print(
                    f"  Protein rows: "
                    f"{len(dataframe):,}"
                )

                print(
                    "  Hyperglycemia missingness: "
                    f"{hyperglycemia_score['missingness_percent']:.2f}%"
                )

                print(
                    "  Mitochondrial-myopathy missingness: "
                    f"{mitochondrial_score['missingness_percent']:.2f}%"
                )

                print(
                    f"  DataFrame available as: "
                    f"dataframes['{accession}']"
                )

            except Exception as error:

                print(
                    f"  FAILED: {error}"
                )

    finally:
        session.close()

    # ========================================================
    # COMBINE PROTEIN-LEVEL PRESENCE RESULTS
    # ========================================================

    combined_tables = []

    presence_columns = (
        METADATA_COLUMNS
        + [
            "Presence_count",
            "Total_LFQ_columns",
            "Presence",
            "Presence_percent",
        ]
    )

    for accession, dataframe in dataframes.items():

        table = dataframe[
            presence_columns
        ].copy()

        table.insert(
            0,
            "PRIDE_accession",
            accession,
        )

        combined_tables.append(
            table
        )

    if combined_tables:

        combined_presence = pd.concat(
            combined_tables,
            ignore_index=True,
        )

    else:
        combined_presence = pd.DataFrame()

    # ========================================================
    # STORE THE TWO MISSINGNESS SCORES PER DATASET
    # ========================================================

    dataset_missingness_scores = pd.DataFrame(
        missingness_results
    )

    missingness_output_file = (
        OUTPUT_DIR
        / "dataset_missingness_scores.csv"
    )

    if not dataset_missingness_scores.empty:

        dataset_missingness_scores.to_csv(
            missingness_output_file,
            index=False,
        )

        print(
            "\nMissingness report saved to:"
        )

        print(
            missingness_output_file
        )

    return (
        dataframes,
        combined_presence,
        dataset_missingness_scores,
    )


(
    dataframes,
    combined_presence,
    dataset_missingness_scores,
) = main()


# Examples:
#
# View the two missingness scores for every dataset:
# print(
#     dataset_missingness_scores[
#         [
#             "PRIDE_accession",
#             "hyperglycemia_missingness_percent",
#             "mitochondrial_myopathy_missingness_percent",
#         ]
#     ]
# )
#
# Get one protein-level DataFrame:
# df = dataframes["PXD010489"]
#
# Proteins detected in more than 10% of LFQ samples:
# detected = df[df["Presence"] > 0.1]


Hyperglycemia feature genes: 3505
Mitochondrial myopathy feature genes: 2315

[1/30] PXD012460
  PRIDE page 0: 73 files
  PRIDE page 1: 0 files
  PRIDE files: 73
  Selected: proteinGroups.txt
  LFQ columns: 36
  Gene column detected: Gene names
  FASTA header column(s) detected: Fasta headers
  Majority protein IDs column detected: Majority protein IDs
  Mapping Majority protein IDs through UniProt: 60 unique accessions
  Rows resolved from Majority protein IDs: 38/38
  Protein rows: 6,033
  Hyperglycemia missingness: 60.88%
  Mitochondrial-myopathy missingness: 54.47%
  DataFrame available as: dataframes['PXD012460']

[2/30] PXD012650
  PRIDE page 0: 100 files
  PRIDE page 1: 1 files
  PRIDE page 2: 0 files
  PRIDE files: 101
  Selected: proteinGroups.txt
  LFQ columns: 24
  Gene column detected: Gene names
  FASTA header column(s) detected: Fasta headers
  Majority protein IDs column detected: Majority protein IDs
  Mapping Majority protein IDs through UniProt: 213 unique accessions


/var/folders/vj/tlj0y71x381bb5cfc_8ylv2nf7_f43/T/ipykernel_38997/751410882.py:1326: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataframe["Gene matching source"] = (
/var/folders/vj/tlj0y71x381bb5cfc_8ylv2nf7_f43/T/ipykernel_38997/751410882.py:1364: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataframe["Presence_count"] = (
/var/folders/vj/tlj0y71x381bb5cfc_8ylv2nf7_f43/T/ipykernel_38997/751410882.py:1368: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, wh

  Protein rows: 3,700
  Hyperglycemia missingness: 71.90%
  Mitochondrial-myopathy missingness: 64.28%
  DataFrame available as: dataframes['PXD045733']

[25/30] PXD046940
  PRIDE page 0: 21 files
  PRIDE page 1: 0 files
  PRIDE files: 21
  Selected: proteinGroups.txt
  LFQ columns: 20
  Gene column detected: Gene names
  FASTA header column(s) detected: Fasta headers
  Majority protein IDs column detected: Majority protein IDs
  Mapping Majority protein IDs through UniProt: 252 unique accessions
  Rows resolved from Majority protein IDs: 0/202
  Protein rows: 5,698
  Hyperglycemia missingness: 63.34%
  Mitochondrial-myopathy missingness: 57.24%
  DataFrame available as: dataframes['PXD046940']

[26/30] PXD047075
  PRIDE page 0: 25 files
  PRIDE page 1: 0 files
  PRIDE files: 25
  Selected: proteinGroups.txt
  LFQ columns: 24
  FASTA header column(s) detected: Fasta headers
  Majority protein IDs column detected: Majority protein IDs
  Mapping Majority protein IDs through UniProt: 2 un

In [57]:
accession = "PXD022985"
session = create_session()

records = get_pride_files(
    accession,
    session,
)

if not records:
    raise FileNotFoundError(
        "PRIDE returned no public files."
    )

print(
    f"  PRIDE files: {len(records)}"
)

candidates = []


  PRIDE files: 100


In [43]:
Genes_present = dataframes["PXD022985"][dataframes["PXD022985"]["Presence"]>0.1]

Genes_present = (
    dataframes["PXD046940"]["Gene names"]
    .dropna()
    .str.split(";")
    .explode()
    .str.strip()
    .dropna()
    .unique()
)
print(Genes_present)
print("Number of unique genes:", len(Genes_present))

KeyError: 'PXD022985'

In [32]:
hyperglycemia_features = read_feature_genes(
    HYPERGLYCEMIA_FEATURES_FILE
)

intersection = set(Genes_present) & set(hyperglycemia_features)
print(len(intersection))

1285
